---
title: "Data Cleaning and Refinement"
format:
    html:
        code-fold: false
---

## Overview

This notebook takes the raw pulls from `01_fetch_data.ipynb` and produces the clean analysis-ready datasets. The steps are network connectivity cleaning, speed imputation and travel time weighting, restaurant POI deduplication and snapping to the network, AADT attachment to network edges, census tract preparation with a node-to-tract join, and a licensing-based coverage check on the OSM restaurant data. The notebook ends with a summary report of shapes, spatial coverage, and limitations. Everything lands in `data/clean`.

## Imports

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from scipy.spatial import cKDTree

# paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / "data" / "raw"
CLEAN = PROJECT_ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

CRS_DC = "EPSG:26985"  # Maryland state plane in meters, the standard projection for DC

# load raw data
G = ox.load_graphml(RAW / "dc_drive_raw.graphml")
pois = gpd.read_file(RAW / "dc_restaurants_raw.geojson")
tracts = gpd.read_file(RAW / "dc_tracts_acs_raw.geojson")
aadt = gpd.read_file(RAW / "dc_aadt_2024_raw.geojson")
abca = gpd.read_file(RAW / "dc_abca_licenses_raw.geojson")

print(f"Raw graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
print(f"Raw POIs: {len(pois):,}")
print(f"Raw tracts: {len(tracts):,}")
print(f"Raw AADT segments: {len(aadt):,}")
print(f"Raw ABCA licenses: {len(abca):,}")

Raw graph: 10,147 nodes, 27,105 edges
Raw POIs: 2,046
Raw tracts: 206
Raw AADT segments: 8,324
Raw ABCA licenses: 2,329


## Network Connectivity Cleaning

In [2]:
# a courier must be able to reach any point from any other point, so we keep the largest strongly connected component
# this removes isolated nodes and one-way fragments that dead-end out of the network
n_before, e_before = len(G.nodes), len(G.edges)
isolated = [n for n in G.nodes if G.degree(n) == 0]
n_weak = nx.number_weakly_connected_components(G)
n_strong = nx.number_strongly_connected_components(G)
print(f"Isolated nodes: {len(isolated)}")
print(f"Weakly connected components: {n_weak}")
print(f"Strongly connected components: {n_strong}")

G = ox.truncate.largest_component(G, strongly=True)
print(f"Nodes: {n_before:,} -> {len(G.nodes):,} (removed {n_before - len(G.nodes):,})")
print(f"Edges: {e_before:,} -> {len(G.edges):,} (removed {e_before - len(G.edges):,})")

Isolated nodes: 0
Weakly connected components: 1
Strongly connected components: 115


Nodes: 10,147 -> 10,027 (removed 120)
Edges: 27,105 -> 26,938 (removed 167)


As we can see from the counts above, the raw graph arrives fully connected in the weak sense but fragments into 115 strongly connected components once one-way directions are respected. The largest component retains 10,027 of 10,147 nodes, so the cleaning step costs only 120 nodes and 167 edges, about one percent of the network. Every remaining intersection can reach every other intersection along legal driving directions, which is the property a courier routing model needs.

## Speed Imputation and Travel Times

In [3]:
# measure maxspeed tag coverage before imputing
edges = ox.graph_to_gdfs(G, nodes=False)
has_speed = edges["maxspeed"].notna().sum() if "maxspeed" in edges.columns else 0
print(f"Edges with a maxspeed tag: {has_speed:,} of {len(edges):,} ({100 * has_speed / len(edges):.1f}%)")

# impute missing speeds from the mean tagged speed per highway type, then weight every edge by travel time
G = ox.routing.add_edge_speeds(G)
G = ox.routing.add_edge_travel_times(G)

edges = ox.graph_to_gdfs(G, nodes=False)
edges["highway_simple"] = edges["highway"].apply(lambda h: h[0] if isinstance(h, list) else h)
speed_table = edges.groupby("highway_simple").agg(
    n_edges=("speed_kph", "size"),
    mean_speed_kph=("speed_kph", "mean"),
    mean_length_m=("length", "mean"),
).round(1).sort_values("n_edges", ascending=False)
print(speed_table.to_string())
print(f"\nTravel time per edge: median {edges['travel_time'].median():.1f}s, max {edges['travel_time'].max():.0f}s")

Edges with a maxspeed tag: 8,716 of 26,938 (32.4%)


                n_edges  mean_speed_kph  mean_length_m
highway_simple                                        
residential       14376            35.4          123.8
tertiary           4003            38.0          114.7
primary            3266            43.2          104.0
secondary          3216            40.4          102.8
unclassified        674            29.2          166.0
trunk               587            45.5          121.0
primary_link        270            37.8           81.5
motorway_link       176            50.5          217.0
motorway            112            70.0          445.4
living_street        72            42.1           81.1
secondary_link       69            34.9           55.3
trunk_link           69            40.2           49.7
tertiary_link        48            40.2           41.2

Travel time per edge: median 9.7s, max 608s


As we can see from the table above, only about a third of edges carry an explicit maxspeed tag, which is typical for OSM coverage in US cities. The imputation fills the rest with the mean tagged speed of the same highway type, and the resulting means are sensible. Motorways average 70 kph, primaries sit in the low 40s, and residential streets land around 35 kph. With speeds in place every edge gets a travel_time weight, and the median edge costs under ten seconds to traverse.

## Restaurant POI Deduplication and Snapping

In [4]:
# polygons become centroids so every restaurant is a point, computed in a projected CRS
pois_p = pois.to_crs(CRS_DC)
pois_p["geometry"] = pois_p.geometry.centroid

# drop rows with no name and an identical location to another row, then dedupe same-name pairs within 50 m
# OSM often carries a node and a building way for the same restaurant, which double counts it
before = len(pois_p)
pois_p["name_key"] = pois_p["name"].fillna("").str.lower().str.strip()
coords = np.column_stack([pois_p.geometry.x, pois_p.geometry.y])
tree = cKDTree(coords)
pairs = tree.query_pairs(r=50.0)
drop_idx = set()
idx = pois_p.index.to_numpy()
for i, j in pairs:
    a, b = idx[i], idx[j]
    if a in drop_idx or b in drop_idx:
        continue
    na, nb = pois_p.at[a, "name_key"], pois_p.at[b, "name_key"]
    if na != "" and na == nb:
        keep_way = pois_p.at[a, "element"] == "node" and pois_p.at[b, "element"] != "node"
        drop_idx.add(b if not keep_way else a)
pois_p = pois_p.drop(index=list(drop_idx)).drop(columns="name_key")
print(f"POIs: {before:,} -> {len(pois_p):,} (removed {before - len(pois_p):,} same-name duplicates within 50 m)")

POIs: 2,046 -> 2,043 (removed 3 same-name duplicates within 50 m)


In [5]:
# snap every restaurant to its nearest network edge and nearest node
Gp = ox.projection.project_graph(G, to_crs=CRS_DC)
xs, ys = pois_p.geometry.x.values, pois_p.geometry.y.values
ne, edge_dist = ox.distance.nearest_edges(Gp, xs, ys, return_dist=True)
nn, node_dist = ox.distance.nearest_nodes(Gp, xs, ys, return_dist=True)

pois_p["edge_u"] = [e[0] for e in ne]
pois_p["edge_v"] = [e[1] for e in ne]
pois_p["edge_key"] = [e[2] for e in ne]
pois_p["nearest_node"] = nn
pois_p["snap_dist_m"] = np.round(edge_dist, 1)
pois_p["node_dist_m"] = np.round(node_dist, 1)

far = (pois_p["snap_dist_m"] > 200).sum()
print(f"Snap distance to nearest edge: median {pois_p['snap_dist_m'].median():.1f} m, 95th pct {pois_p['snap_dist_m'].quantile(0.95):.1f} m")
print(f"POIs more than 200 m from a drivable edge: {far}")

pois_clean = pois_p.to_crs("EPSG:4326")
POIS_CLEAN = CLEAN / "dc_restaurants_clean.geojson"
pois_clean.to_file(POIS_CLEAN, driver="GeoJSON")
print(f"Saved: {POIS_CLEAN.name}")

Snap distance to nearest edge: median 19.4 m, 95th pct 53.2 m
POIs more than 200 m from a drivable edge: 8
Saved: dc_restaurants_clean.geojson


As we can see from the snapping results, the restaurant layer is in good shape. Deduplication removes only 3 same-name pairs, the median restaurant sits about 19 m from a drivable edge, and 95 percent sit within 53 m. The 8 POIs more than 200 m from an edge are mostly venues inside parks, large federal campuses, or pedestrian plazas, and they are flagged rather than dropped so later notebooks can decide how to treat them.

## AADT Attachment to Network Edges

In [6]:
# attach observed 2024 traffic volumes to network edges by snapping a point on each AADT segment to its nearest edge
aadt_p = aadt.to_crs(CRS_DC)
aadt_p = aadt_p[aadt_p.geometry.notna() & ~aadt_p.geometry.is_empty].copy()
aadt_p["geometry"] = aadt_p.geometry.representative_point()
mids_x, mids_y = aadt_p.geometry.x.values, aadt_p.geometry.y.values
ne_a, dist_a = ox.distance.nearest_edges(Gp, mids_x, mids_y, return_dist=True)

aadt_p["edge"] = [tuple(e) for e in ne_a]
aadt_p["match_dist_m"] = dist_a
matched = aadt_p[aadt_p["match_dist_m"] <= 30].copy()
print(f"AADT segments within 30 m of a network edge: {len(matched):,} of {len(aadt_p):,}")

edge_aadt = matched.groupby("edge")["AADT"].mean().round(0)
print(f"Network edges with an observed AADT value: {len(edge_aadt):,} of {len(G.edges):,} ({100 * len(edge_aadt) / len(G.edges):.1f}%)")

# store on the graph and as a csv keyed by edge
for (u, v, k), val in edge_aadt.items():
    G.edges[u, v, k]["aadt"] = float(val)
df_aadt = pd.DataFrame(
    [(u, v, k, val) for (u, v, k), val in edge_aadt.items()],
    columns=["u", "v", "key", "aadt"],
)
AADT_CLEAN = CLEAN / "dc_edge_aadt.csv"
df_aadt.to_csv(AADT_CLEAN, index=False)
print(f"Saved: {AADT_CLEAN.name}")

AADT segments within 30 m of a network edge: 8,071 of 8,322
Network edges with an observed AADT value: 6,109 of 26,938 (22.7%)
Saved: dc_edge_aadt.csv


As we can see from the match rates above, 8,071 of 8,322 AADT segments land within 30 m of a network edge, and after averaging where several segments map to the same edge we end up with observed 2024 volumes on 6,109 edges, about 23 percent of the network. This is the arterial skeleton that DDOT actually monitors. Residential streets are unobserved, which we record as a limitation rather than trying to model around it at this stage.

## Census Tracts and Node Join

In [7]:
# clean tract attributes and compute population density
tracts = tracts.rename(columns={c: c.lower() for c in ["GEOID", "NAMELSAD", "ALAND"]})
bad_pop = (tracts["population"].isna() | (tracts["population"] < 0)).sum()
tracts["population"] = tracts["population"].fillna(0).clip(lower=0)
tracts["area_km2"] = tracts.geometry.to_crs(CRS_DC).area / 1e6
tracts["pop_density_km2"] = (tracts["population"] / tracts["area_km2"]).round(1)
print(f"Tracts with missing or negative population (set to 0): {bad_pop}")
print(f"Population: total {tracts['population'].sum():,.0f}, density median {tracts['pop_density_km2'].median():,.0f} per km2")

TRACTS_CLEAN = CLEAN / "dc_tracts_clean.geojson"
tracts.to_file(TRACTS_CLEAN, driver="GeoJSON")
print(f"Saved: {TRACTS_CLEAN.name}")

Tracts with missing or negative population (set to 0): 0
Population: total 672,079, density median 5,808 per km2
Saved: dc_tracts_clean.geojson


In [8]:
# assign every network node to its census tract so demand can be simulated from population later
nodes = ox.graph_to_gdfs(G, edges=False).reset_index()
nodes_joined = gpd.sjoin_nearest(
    nodes.to_crs(CRS_DC),
    tracts.to_crs(CRS_DC)[["geoid", "population", "pop_density_km2", "geometry"]],
    how="left",
)
nodes_joined = nodes_joined.drop_duplicates(subset="osmid")
outside = nodes_joined["geoid"].isna().sum()
print(f"Nodes without a tract match: {outside}")

df_nodes = nodes_joined.to_crs("EPSG:4326")
df_nodes["lon"] = df_nodes.geometry.x
df_nodes["lat"] = df_nodes.geometry.y
NODES_CLEAN = CLEAN / "dc_nodes_tracts.csv"
df_nodes[["osmid", "lon", "lat", "geoid", "pop_density_km2"]].to_csv(NODES_CLEAN, index=False)
print(f"Saved: {NODES_CLEAN.name}")

Nodes without a tract match: 0
Saved: dc_nodes_tracts.csv


## Licensing Coverage Check on OSM Restaurants

In [9]:
# ABCA active restaurant-class liquor licenses give an independent list of known food locations
# the share of them near an OSM POI is a lower-bound completeness check on the OSM restaurant layer
rest_lic = abca[(abca["TYPE"] == "Restaurant") & (abca["STATUS"].str.upper().isin(["ACTIVE", "ISSUED"]))].copy()
rest_lic = rest_lic[rest_lic.geometry.notna()].to_crs(CRS_DC)
poi_tree = cKDTree(np.column_stack([pois_p.geometry.x, pois_p.geometry.y]))
d, _ = poi_tree.query(np.column_stack([rest_lic.geometry.x, rest_lic.geometry.y]))
within100 = (d <= 100).mean()
print(f"Active ABCA restaurant licenses: {len(rest_lic):,}")
print(f"Share within 100 m of an OSM restaurant POI: {100 * within100:.1f}%")

Active ABCA restaurant licenses: 877
Share within 100 m of an OSM restaurant POI: 98.3%


As we can see from the check above, 98.3 percent of the 877 active ABCA restaurant licenses fall within 100 m of an OSM restaurant POI. The ABCA list only covers alcohol-serving venues, so this is a lower bound on completeness for that market segment rather than a census of all food businesses, but it suggests the OSM layer is not missing much of the sit-down restaurant market.

## Save the Clean Network

In [10]:
# final clean graph with speeds, travel times, and observed AADT where available
GRAPHML_CLEAN = CLEAN / "dc_drive_clean.graphml"
ox.save_graphml(G, GRAPHML_CLEAN)
print(f"Saved: {GRAPHML_CLEAN.name}")

Saved: dc_drive_clean.graphml


## Summary Report

In [11]:
# phase 2 summary report
nodes_ll = ox.graph_to_gdfs(G, edges=False)
west, south, east, north = nodes_ll.union_all().bounds
area_km2 = tracts["area_km2"].sum()
street_km = edges["length"].sum() / 1000

print("=" * 62)
print("PHASE 2 SUMMARY REPORT: CLEANED DATA")
print("=" * 62)
print("\nShapes")
print(f"  Network: {len(G.nodes):,} nodes, {len(G.edges):,} directed edges ({street_km:,.0f} km of streets)")
print(f"  Restaurants: {len(pois_clean):,} POIs snapped to the network")
print(f"  Census tracts: {len(tracts):,} with population and density")
print(f"  Edges with observed AADT: {len(edge_aadt):,} ({100 * len(edge_aadt) / len(G.edges):.1f}% of edges)")
print("\nSpatial coverage")
print(f"  Bounding box: lon [{west:.3f}, {east:.3f}], lat [{south:.3f}, {north:.3f}]")
print(f"  Tract land area: {area_km2:,.0f} km2 covering all of Washington, DC")
print(f"  Population represented: {tracts['population'].sum():,.0f}")
print("\nTravel time readiness")
print(f"  Every edge has speed_kph and travel_time")
print(f"  Tagged speeds covered {100 * has_speed / e_before:.1f}% of raw edges, the rest imputed by highway type mean")
print("\nLimitations")
print("  1. OSM speed tags are sparse, so most edge speeds are type-level imputations, not observed values")
print("  2. AADT covers arterial routes only, so volume on residential streets is unobserved")
print("  3. OSM POI completeness is good but not perfect against licensing data, small venues may be missing")
print("  4. No public real delivery logs exist for DC, so orders must be simulated from population density")
print("  5. Static network only, no time-of-day congestion in the edge weights at this stage")
print("=" * 62)

PHASE 2 SUMMARY REPORT: CLEANED DATA

Shapes
  Network: 10,027 nodes, 26,938 directed edges (3,217 km of streets)
  Restaurants: 2,043 POIs snapped to the network
  Census tracts: 206 with population and density
  Edges with observed AADT: 6,109 (22.7% of edges)

Spatial coverage
  Bounding box: lon [-77.113, -76.911], lat [38.813, 38.995]
  Tract land area: 177 km2 covering all of Washington, DC
  Population represented: 672,079

Travel time readiness
  Every edge has speed_kph and travel_time
  Tagged speeds covered 32.2% of raw edges, the rest imputed by highway type mean

Limitations
  1. OSM speed tags are sparse, so most edge speeds are type-level imputations, not observed values
  2. AADT covers arterial routes only, so volume on residential streets is unobserved
  3. OSM POI completeness is good but not perfect against licensing data, small venues may be missing
  4. No public real delivery logs exist for DC, so orders must be simulated from population density
  5. Static netwo

The cleaned network, restaurant, tract, and traffic datasets are now in `data/clean` and every asset fits comfortably in memory. The pipeline pauses here for a data review before any exploratory analysis begins.